# 第18章 可解释性人工智能 (eXplainable AI, XAI)

> "如果AI决定不给你贷款，你至少有权利知道为什么。"

## 1. 知识地图：章节结构与概览

```
可解释性AI (XAI)
├── 18.1 可解释性人工智能的重要性
│   ├── 为什么需要XAI？
│   │   ├── 银行放贷：法律规定必须给出理由
│   │   ├── 医疗诊断：人命关天
│   │   ├── 法律判决：是否公正/种族歧视
│   │   ├── 自动驾驶：急刹车的理由是什么
│   │   └── 模型调试：发现学到错误特征
│   ├── "聪明汉斯"故事（马算数靠读人类微表情）
│   └── 线性模型 → 可解释但弱 → 深度模型 → 强但黑盒 → XAI的目标
├── 18.2 决策树模型的可解释性
│   ├── 决策树天然可解释（节点=问题，路径=规则）
│   ├── 复杂决策树也是黑盒
│   ├── 随机森林 = 多棵决策树 → 更难解释
│   └── Kaggle比赛常胜 ≠ 推理最佳选择
├── 18.3 可解释性机器学习的目标
│   ├── 完全理解模型 vs 人类能接受的解释
│   └── 心理学实验（94% vs 93% 接受率）
├── 18.4 局部解释 (Local Explanation)
│   ├── 遮盖法：遮挡不同区域看影响
│   ├── 显著性图 (Saliency Map)
│   │   ├── 原理：∂e/∂x_i → 梯度大小 = 重要性
│   │   ├── 梯度饱和问题
│   │   └── 真实案例：模型看水印而不是看马
│   ├── SmoothGrad：多次加噪取平均
│   ├── 积分梯度 (Integrated Gradients)
│   ├── 可视化隐层：PCA/t-SNE降维
│   ├── 探针 (Probe)
│   │   ├── 训练简单分类器探查隐层含什么信息
│   │   └── 语音合成的TTS探针：验证去噪效果
│   └── LIME：局部可解释模型无关解释
├── 18.5 全局解释 (Global Explanation)
│   ├── 滤波器可视化：找最大化某滤波器激活的输入
│   │   └── X* = argmax Σ a_ij（梯度上升）
│   ├── 类别可视化：找最能代表某类的图像
│   │   ├── 直接优化 → 噪声（对抗样本）
│   │   └── 加正则项 R(X) → 更像真实图像
│   └── 用生成器约束：z* = argmax Score(G(z))
└── 18.6 扩展与小结
    ├── 可解释模型模拟黑盒模型
    └── LIME：局部线性近似
```

## 2. 为什么可解释性AI如此重要？

### 2.1 四个真实应用场景

| 场景 | 为什么需要解释 | 可能的后果 |
|------|------|------|
| **银行贷款** | 法律要求：拒绝贷款必须说明理由 | 被起诉歧视 |
| **医疗诊断** | 人命关天：医生需要理解模型的判断依据 | 误诊导致死亡 |
| **法律判决** | 需验证模型是否公正（无种族/性别歧视） | 冤假错案 |
| **自动驾驶** | 急刹车是因为看到老人过马路，还是故障？ | 人身伤害 |

### 2.2 "聪明汉斯"——AI界的寓言

一匹叫汉斯的马，似乎会算术。问它 $\sqrt{9}$ 是多少，它会用马蹄跺3下。但真相是：
- 汉斯不会算术
- 它读人类围观者微妙的表情变化
- 当跺到正确答案时，围观者不自觉放松/微笑 → 汉斯知道该停了

**深度学习模型可能也是"聪明汉斯"：**
- 一个马的分类器在ImageNet上表现很好
- 但检查显著性图发现：它根本没看马！
- 它在看图片左下角的英文水印——因为训练集中"马"的照片都来自同一个摄影网站

**教训：模型给出的正确答案，未必来自正确的推理。XAI帮我们发现这类问题。**

### 2.3 线性模型 vs 深度模型的解释性困境

- 线性模型：可解释性强（看权重就知道每个特征的重要性），但表达能力弱
- 深度模型：表达能力强，但是黑盒
- 正确态度：不是放弃深度模型，而是**让深度模型也变得可解释**

## 3. 可解释性的目标：人类需要什么样的解释？

### 3.1 1970年哈佛大学心理学实验

在哈佛图书馆排队复印时，如果有人插队：
- 直接说"让我先印5页"：60%的人同意
- 说"让我先印，因为我赶时间"：94%的人同意
- 说"让我先印，因为我需要先印"：93%的人同意！

**结论**：人类需要的不是"真正的原因"，而是一个**听起来合理的理由**。

### 3.2 对XAI的启示

- 好的解释 = 人类能接受的解释
- 我们不一定需要完全理解模型的每一个内部运作
- 人脑也是黑盒，但我们相信别人的判断——因为有解释

这也是为什么"用生成器约束可视化结果"能产生更受欢迎的解释——因为我们主观上更容易接受"像真实图像的"解释。

## 4. 局部解释：为什么你认为这张图片是猫？

### 4.1 遮盖法（最朴素的方法）

用一个灰色方块在图片上滑动（遮挡不同区域），看模型分类结果如何变化：
- 遮挡狗的**脸部** → 模型不再认为是狗 → 脸部是关键
- 遮挡狗的**背景** → 模型仍认为是狗 → 背景不重要

### 4.2 显著性图 (Saliency Map)

**原理：** 对输入 $x$（$x_1$ 到 $x_N$ 代表N个像素），计算损失 $e$（交叉熵）关于每个像素的偏导数：

$$\text{重要性}(x_i) \propto \left|\frac{\partial e}{\partial x_i}\right|$$

梯度大的像素 = 对决策影响大 = 重要。将所有像素的梯度可视化就得到显著性图。

**公式说明：**
- $e$：将图片输入模型后的交叉熵损失（越大=识别越差）
- $\frac{\partial e}{\partial x_i}$：损失对像素 $x_i$ 的偏导数
- 如果微调 $x_i$ 后损失变化大 → 该像素对分类重要

### 4.3 梯度饱和问题

象鼻长度判断"大象"：
- 鼻子短的时候：越长越像大象（梯度大）
- 鼻子已经够长了：再长也不会更像（梯度 ≈ 0！）
- 但鼻子长度**仍然是判断大象的关键指标**

**问题：** 梯度为零不代表特征不重要——只是进入了饱和区域！

### 4.4 SmoothGrad

解决方法：对图像多次加噪声，每次计算梯度，最后取平均：

$$\text{SmoothGrad}(x) = \frac{1}{N} \sum_{i=1}^{N} \nabla f(x + \mathcal{N}(0, \sigma^2))$$

- 噪声让模型"离开"饱和区域
- 多次平均消除噪声引入的随机性

In [ ]:
# 显著性图和 SmoothGrad 的 PyTorch 实现
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_saliency_map(model, input_image, target_class):
    """
    计算显著性图
    原理：|∂L/∂x_i| 越大 → 像素 i 越重要
    """
    input_image.requires_grad = True
    output = model(input_image)
    loss = output[0, target_class]  # 目标类的logit
    model.zero_grad()
    loss.backward()
    # 取梯度的绝对值作为重要性
    saliency = input_image.grad.abs()
    # 如果是RGB图，取各通道最大值
    if saliency.size(1) == 3:
        saliency = saliency.max(dim=1, keepdim=True)[0]
    return saliency

def smooth_grad(model, input_image, target_class, n_samples=50, noise_std=0.15):
    """
    SmoothGrad：多次加噪求平均
    解决梯度饱和问题
    """
    saliency_sum = 0.0
    for _ in range(n_samples):
        noise = torch.randn_like(input_image) * noise_std
        noisy_input = (input_image + noise).detach().requires_grad_(True)
        output = model(noisy_input)
        loss = output[0, target_class]
        model.zero_grad()
        loss.backward()
        saliency_sum += noisy_input.grad.abs()
    return saliency_sum / n_samples

print("显著性图计算流程：")
print("1. 输入: 图像 x, 目标类别 c")
print("2. 前向: output = model(x), loss = output[c]")
print("3. 反向: loss.backward()")
print("4. 显著性图 = |grad| = |∂loss/∂x|")
print()
print("SmoothGrad改进：通过加噪避免梯度饱和")

In [ ]:
# 探针方法 (Probing) 的 PyTorch 实现
import torch
import torch.nn as nn

class ProbeClassifier(nn.Module):
    """
    探针分类器
    用于探查BERT/dNN隐层包含什么语言信息
    """
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # 探针必须简单！否则结论不可靠
        self.classifier = nn.Linear(input_dim, num_classes)
    
    def forward(self, hidden_states):
        return self.classifier(hidden_states)

def probe_analysis(pretrained_model, probe, dataloader):
    """
    探针分析流程：
    1. 冻结预训练模型
    2. 用探针分类器预测语言属性（如POS词性）
    3. 探针准确率 = 隐层包含该信息的程度
    """
    pretrained_model.eval()
    probe.train()
    optimizer = torch.optim.Adam(probe.parameters(), lr=0.001)
    
    for batch_x, batch_y in dataloader:
        with torch.no_grad():
            hidden = pretrained_model.get_hidden_states(batch_x)
        logits = probe(hidden)
        loss = nn.functional.cross_entropy(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

print("探针方法的使用场景：")
print("1. 探针=POS分类器 → 准确率高 = 隐层含词性信息")
print("2. 探针=NER分类器 → 准确率高 = 隐层含命名实体信息")
print("3. 探针=TTS模型 → 能否还原声音 = 隐层含讲者信息")
print()
print("警告：探针太强 → 可能学到探针自己的能力（非隐层的信息）")
print("警告：探针太弱 → 无法正确反映隐层信息（训练不足）")

## 5. 网络内部探查：窥视黑盒

### 5.1 可视化隐层表示

语音识别案例（Hinton的工作）：
- 输入层(MFCC)：不同人说同样的话 → 在MFCC空间完全混在一起（看不出共性）
- 第8隐藏层：同样的句子内容聚在一起了！
- 结论：网络通过层层处理，自动学会了"把内容从讲者特征中分离出来"

**方法**：用 PCA 或 t-SNE 将高维隐层向量降维到2D，然后画散点图观察。

### 5.2 探针方法 (Probing)

用一个简单的分类器（通常只是线性层）来"探测"BERT某层是否包含某种信息：

1. 将BERT某层的输出（embedding）喂给探针
2. 探针尝试预测语言属性（如POS词性、NER命名实体）
3. 探针准确率 = 该层包含该信息的程度

**陷阱：**
- 探针太强 → 可能自己学会了目标任务（不能归因于隐层信息）
- 探针太弱/训练不充分 → 误判为"隐层没有该信息"

### 5.3 语音合成的TTS探针

将网络的隐层向量喂给TTS模型，尝试还原原始声音：
- 如果还原出来有讲者的声音特征 → 该层仍保留了讲者信息
- 如果还原出来内容对但听不出是谁 → 该层成功去除了讲者特征
- 实验发现：经过5层BiLSTM后，不同讲者的声音变成了同一种"机器声"

## 6. 全局解释：这个模型心中的"猫"长什么样？

### 6.1 滤波器可视化

对于卷积网络的某个滤波器，找最大化它激活的图像：

$$X^* = \arg\max_X \sum_{i,j} a_{ij}$$

- $a_{ij}$：滤波器输出特征图中位置 $(i,j)$ 的值
- 用梯度上升法（不是梯度下降！）求解 $X^*$
- MNIST手写数字识别：第二层卷积的滤波器检测的是横线、竖线、斜线等基本笔画

### 6.2 类别可视化（直接优化方式）

$$X^* = \arg\max_X \text{Score}(\text{类别k} | X)$$

不加约束直接优化 → 得到的是人眼看不懂的噪声图案（类似于对抗样本）→ 但机器认作是该类！

### 6.3 加入约束

$$X^* = \arg\max_X \left[\text{Score}(\text{类别k} | X) + R(X)\right]$$

其中 $R(X)$ 是正则项，例如：
- 像素值平方和（抑制过大像素值）
- 梯度的平方和（鼓励平滑）
- 总变差（Total Variation）正则化

### 6.4 用生成器约束

用GAN/VAE的生成器 $G(z)$ 产生图像，再输入分类器：

$$z^* = \arg\max_z \text{Score}(\text{类别k} | G(z))$$

- 因为生成器只能产生"像真实图像"的输出
- 天然约束了解空间
- 产生的可视化结果更接近人类认知

但这里有个哲学问题：我们用生成器约束是为了得到"人类觉得舒服"的结果——这是真正的解释，还是只是让人类开心的解释？

In [ ]:
# 滤波器可视化的 PyTorch 实现
import torch
import torch.nn as nn

def visualize_filter(model, layer_name, filter_index, input_shape=(1, 3, 224, 224), steps=100, lr=0.1):
    """
    滤波器可视化：找一个输入最大化指定滤波器的激活
    使用梯度上升法（注意是上升！）
    """
    # 随机初始化一张图
    x = torch.randn(input_shape, requires_grad=True)
    
    for step in range(steps):
        # 前向传播到目标层
        feature_maps = model.get_layer_output(x, layer_name)
        
        # 目标：最大化该滤波器的激活值总和
        loss = feature_maps[0, filter_index].sum()
        
        # 梯度上升（注意是 + 不是 -）
        grads = torch.autograd.grad(loss, x)[0]
        x.data += lr * grads
        
        # 归一化到合理范围
        x.data = torch.clamp(x.data, 0, 1)
    
    return x.detach()

print("滤波器可视化步骤：")
print("1. 随机初始化图像 X ~ N(0,1)")
print("2. 计算目标滤波器特征图: A = model.get_feature_map(X)")
print("3. 最大化: loss = sum(A[filter_index])")
print("4. 梯度上升: X = X + lr * ∂loss/∂X")
print("5. 重复100步 → 得到的X就是滤波器'喜欢'的模式")

In [ ]:
# LIME（局部可解释模型无关解释）的简化实现
import torch
import numpy as np
from sklearn.linear_model import LinearRegression

def lime_explain(model, sample, num_features, num_samples=1000):
    """
    LIME的简化实现
    核心思想：在样本周围用线性模型局部近似复杂模型
    """
    # 1. 在样本周围采样
    perturbations = np.random.binomial(1, 0.5, size=(num_samples, num_features))
    
    # 2. 构造扰动样本（只保留被选中的特征）
    perturbed_samples = []
    for p in perturbations:
        mask = torch.tensor(p, dtype=torch.float32)
        perturbed = sample * mask.unsqueeze(-1)  # 0掉被遮盖的特征
        perturbed_samples.append(perturbed)
    perturbed_samples = torch.stack(perturbed_samples)
    
    # 3. 用黑盒模型预测所有扰动样本
    with torch.no_grad():
        predictions = model(perturbed_samples).numpy()
    
    # 4. 训练线性模型（可解释的）
    # 根据与原始样本的相似度加权
    distances = np.sum(perturbations, axis=1)  # 扰动距离
    weights = np.exp(-distances / num_features)  # 越接近原始样本权重越大
    
    linear_model = LinearRegression()
    linear_model.fit(perturbations, predictions, sample_weight=weights)
    
    # 5. 线性模型的系数 = 各特征的重要性
    feature_importance = linear_model.coef_
    return feature_importance

print("LIME 的核心流程：")
print("1. 在样本x周围采样大量'邻居'（随机遮盖特征）")
print("2. 用黑盒模型标注这些邻居")
print("3. 训练线性模型拟合这些预测")
print("4. 线性模型的权重 = 各特征的重要性")

## 7. 常见误区与易错点

### 误区 1：梯度大 = 特征重要
**纠正：** 梯度饱和问题——特征已经"足够强"时梯度为零，但不代表不重要。需要用积分梯度等方法。

### 误区 2：显著性图的亮点代表模型"看到"了物体
**纠正：** 模型可能在看完全不同的东西（如PASCAL VOC中看水印判断马）。显著性图只能告诉我们"模型关注什么"，不一定是我们以为的物体。

### 误区 3：可解释性 = 完全理解模型内部
**纠正：** 可解释性的目标更务实——提供人类能接受的理由。人脑也是黑盒但我们相信他人的判断。

### 误区 4：决策树天然可解释
**纠正：** 浅层决策树可解释，但深度很大的决策树和随机森林也是黑盒。

### 误区 5：探针准确率低 = 隐层没有该信息
**纠正：** 可能是探针本身训练不足（学习率没调好等）。需要排除探针自身的问题才能下结论。

### 误区 6：可视化出来的"清晰图像"就是模型的真实认知
**纠正：** 我们用了生成器/正则化约束来让结果"好看"——这也许只是让人类满意的样本，不一定是模型"真正的想法"。XAI有很强的主观性。

## 8. 与其他章节的联系

| 章节 | 联系 |
|------|------|
| 第4章 CNN | 滤波器可视化直接解释CNN每层在做什么 |
| 第8章 GAN | GAN生成器用于约束全局解释的可视化结果 |
| 第10章 BERT | 探针方法广泛用于分析BERT各层学到了什么 |
| 第12章 对抗攻击 | 全局解释中无约束优化结果本质上是"对抗样本" |
| 第15章 元学习 | NAS搜索出的架构需要XAI来解释其设计逻辑 |
| 第19章 ChatGPT | 神经编辑和Machine Unlearning是XAI的延伸挑战 |

## 9. 核心公式汇总

### 显著性图

$$\text{Saliency}(x_i) = \left|\frac{\partial L}{\partial x_i}\right|$$

### SmoothGrad

$$\text{SmoothGrad}(x) = \frac{1}{N} \sum_{n=1}^{N} \nabla_x f(x + \epsilon_n), \quad \epsilon_n \sim \mathcal{N}(0, \sigma^2)$$

### 滤波器可视化（梯度上升）

$$X^* = \arg\max_X \sum_{i,j} a_{ij}^{(k)}$$

$a_{ij}^{(k)}$ 是第 $k$ 个滤波器输出特征图中位置 $(i,j)$ 的值。

### 带正则的类别可视化

$$X^* = \arg\max_X \left[y_i(X) + \lambda R(X)\right]$$

### 用生成器约束的可视化

$$z^* = \arg\max_z y_i(G(z))$$

其中 $G(z)$ 是生成器，$y_i$ 是对应类别的分数。

## 10. 关键总结

1. **XAI 不只是一个学术课题**：它在金融、医疗、法律、自动驾驶等场景中是必需品
2. **"聪明汉斯"效应警示**：模型可能学到意想不到的虚假关联（如水印而非物体本身）
3. **显著性图是第一工具**：$|\partial L/\partial x|$ 告诉你模型关注图像的哪些区域
4. **梯度饱和是显著性图的致命缺陷**：用 SmoothGrad 或积分梯度来解决
5. **局部解释 + 全局解释 = 完整的 XAI 工具链**
6. **探针方法很强大但需谨慎**：探针太强/太弱都会导致错误结论
7. **可解释性有主观性**：我们倾向于接受"人类看着舒服"的解释（心理学的发现）
8. **滤波器可视化揭示CNN学到的层次化特征**：浅层=边缘纹理，深层=语义概念
9. **无约束的全局解释 ≈ 对抗样本**：需要正则化或生成器约束
10. **XAI 能帮助我们发现模型漏洞**：如果模型只看水印，那就是该修复的信号